Import all necessary libraries, covering parsing logs,labeling, analysis, recons

In [1]:
import pandas as pd
from pathlib import Path

# -------- parsing ----------
from parsing.syslog import parse_syslog_csv
from parsing.auditd import parse_audit_log
from parsing.auth import parse_auth_log
from parsing.zeek import (
    parse_conn_log, parse_dns_log, parse_http_log, 
    parse_files_log, parse_ssl_log, parse_ssh_log
)
from parsing.suricata import parse_suricata_eve
from parsing.azure_wins import (
    parse_azure_conn, parse_azure_process, parse_azure_security,
    parse_azure_events, parse_azure_port, parse_azure_perf
)
from parsing.tracee import parse_tracee_ndjson

# ---------- labeling / analysis / recons ----------
from labeling.tagger import tag_steps
from analysis.coverage import step_coverage
from analysis.failure_taxonomy import failure_taxonomy 
from analysis.metrics import compute_metrics
from analysis.ambig import ambiguity

from recons.event_graph import build_event_graph
from recons.chain_builder import reconstruct_chain_detailed
# --- scenarios config ---
from scenarios import SCENARIOS



define the mapping from source -> parser

In [2]:
from typing import Callable, List, Tuple, Dict

def _find_files(log_root: Path, patterns: List[str]) -> List[Path]:
    ''' recursively find files matching any of the given patterns in log_root

    '''
    hits = []
    for pat in patterns:
        hits.extend(log_root.rglob(pat) if not pat.startswith("**/") else log_root.rglob(pat[3:]))
    # remove duplicates while preserving order
    uniq = sorted(set([p for p in hits if p.is_file()]))
    return uniq


def _parse_many(files: List[Path], parser: Callable[[str], pd.DataFrame], *, source: str, source_kind: str) -> pd.DataFrame:
    ''' Multiple files are parsed and merged using the same parser, and source fields are added.
    
    '''
    frames = []
    for p in files:
        try:
            df = parser(str(p))
        except Exception as e:
            print(f"Warning: failed to parse {p} with {parser.__name__}")
            print("The reason was:", e)
            continue
        if df is None or len(df) == 0:
            continue
        df = df.copy()
        df["source"] = source
        df["source_kind"] = source_kind
        df["source_file"] = str(p)
        frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def _parse_prefixed_components(
    log_root: Path,
    *,
    prefix: str, # "zeek"/"azure"
    component_parsers: Dict[str, Callable[[str], pd.DataFrame]],
    source: str,
) -> pd.DataFrame:
    '''
    recursively find and parse log files for components with given prefix
    '''
    frames = []
    items = component_parsers.items() if isinstance(component_parsers, dict) else component_parsers

    for comp, fn in items:
        patterns = [
            f"{prefix}_{comp}.csv",
            f"{prefix}_{comp}_*.csv",
            f"{prefix}_{comp}*.csv",
        ]
        files = _find_files(log_root, patterns)
        df = _parse_many(files, fn, source=source, source_kind=f"{source}_{comp}")
        if len(df):
            frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def parse_source(source_key: str, log_root: Path) -> pd.DataFrame:
    '''
    return unified source dataframe for a given source_key
    args:
        log_root: Path to scenario log root
    '''
    # --- syslog ---
    if source_key == "syslog":
        files = _find_files(log_root, ["azure_syslog.csv", "azure_syslog*.csv"])
        return _parse_many(files, parse_syslog_csv, source="syslog", source_kind="syslog")
    
    # --- auditd ---
    if source_key == "auditd":
        files = _find_files(log_root, ["audit.log", "auditd*.log"])
        return _parse_many(files, parse_audit_log, source="auditd", source_kind="auditd")

    # --- auth ---
    if source_key == "auth":
        files = _find_files(log_root, ["auth.log", "auth*.log"])
        return _parse_many(files, parse_auth_log, source="auth", source_kind="auth")

    # ----- suricate -----
    if source_key == "suricata":
        files = _find_files(log_root, ["eve.json", "eve*.json"])
        return _parse_many(files, parse_suricata_eve, source="suricata", source_kind="suricata")

    # ----- tracee -------
    if source_key == "tracee":
        files = _find_files(log_root, ["tracee-events.json"])
        return _parse_many(files, parse_suricata_eve, source="tracee", source_kind="tracee")


    # --- zeek (multi-log) ---
    if source_key == "zeek":
        zeek_parsers: List[Tuple[str, Callable[[str], pd.DataFrame]]] = [
            ("conn",  parse_conn_log),
            ("dns",   parse_dns_log),
            ("http",  parse_http_log),
            ("files", parse_files_log),
            ("ssh",   parse_ssh_log),
            ("ssl",   parse_ssl_log),
        ]
        return _parse_prefixed_components(
            log_root,
            prefix="zeek",
            component_parsers=zeek_parsers,
            source="zeek",
        )

    # --- azure windows ---
    if source_key.startswith("azure_"):
        comp = source_key.replace("azure_", "", 1)  # conn/process/security/events/port/syslog...
        azure_component_parsers = {
            "conn":     parse_azure_conn,
            "process":  parse_azure_process,
            "security": parse_azure_security,
            "events":   parse_azure_events,
            "port":     parse_azure_port,     
            "syslog":   parse_syslog_csv,   
        }
        fn = azure_component_parsers.get(comp)
        if fn is None:
            return pd.DataFrame()

        patterns = [
            f"azure_{comp}.csv",
            f"azure_{comp}_*.csv",
            f"azure_{comp}*.csv",
        ]
        files = _find_files(log_root, patterns)
        return _parse_many(files, fn, source=source_key, source_kind=source_key)

    return pd.DataFrame()


run pipeline (tag + metrics + chain) with given sources

In [3]:
def _normalize_ts(df: pd.DataFrame, ts_col="ts") -> pd.DataFrame:
    if df is None or len(df) == 0 or ts_col not in df.columns:
        return df
    out = df.copy()

    ts = pd.to_datetime(out[ts_col], errors="coerce", utc=True)

    out[ts_col] = ts.dt.tz_convert(None)
    return out

def run_pipeline_for_sources(enabled_sources, log_root: Path):
    # 1) parsing
    frames = []

    for s in enabled_sources:
        df = parse_source(s, log_root)
        if df is not None and len(df) > 0:
            df['source'] = s
            frames.append(df)
    
    events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    # --- guard: prevent cross-scenario leakage ---
    if len(events) and "source_file" in events.columns:
        root = str(Path(log_root).resolve())
        events = events[events["source_file"].astype(str).str.startswith(root)].copy()

    # 2) tagging
    tagged = tag_steps(events) if len(events) else events
    if len(tagged) and "ts" in tagged.columns:
        tagged = tagged.copy()
        tagged["ts"] = pd.to_datetime(tagged["ts"], utc=True, errors="coerce").dt.tz_convert(None)
    # tagged = _normalize_ts(tagged, "ts")

    # 3) analytics
    cov = step_coverage(tagged) if len(tagged) else {}
    fails = failure_taxonomy(tagged) if len(tagged) else {}
    amb = ambiguity(tagged) if len(tagged) else {}

    # 4) reconstruction
    g = None
    chain_steps = []
    if len(tagged) and "step" in tagged.columns and "ts" in tagged.columns:
        g = build_event_graph(tagged)
        d = reconstruct_chain_detailed(g, tagged.dropna(subset=["step"]))
        chain_steps = d["steps"]

    # 5) metrics (chain-aware)
    metrics = compute_metrics(tagged, chain=chain_steps, max_gap_seconds=600) if len(tagged) else {}

    # 6) reconstruction summary
    observed_steps = []
    if len(tagged) and "step" in tagged.columns:
        observed_steps = sorted(tagged["step"].dropna().unique().tolist())

    recon = {
        "n_events": int(len(events)) if len(events) else 0,
        "n_tagged_steps": int(len(observed_steps)),
        "n_chain_steps": int(len(chain_steps)),
        "reconstructability": (len(chain_steps) / len(observed_steps)) if observed_steps else 0.0,
        "observed_steps": observed_steps,
        "chain_steps": chain_steps,
    }

    return {
        "events": events,
        "tagged": tagged,
        "coverage": cov,
        "failures": fails,
        "ambiguity": amb,
        "metrics": metrics,
        "chain": chain_steps,
        "graph": g,
        "recon": recon,
    }
    

### comparison between single source vs multiple sources

- multiple sources: all sources from one scenario
- single source: every source in one scenarios separately runs once, find failure mode and missing part
- output one summary

In [4]:
def get_expected_sources_for_scenario(sc) -> list:
    # sc.expected_sources is dict[str,bool]
    return [k for k, v in sc.expected_sources.items() if v]

def analyze_scenario(scenario_key: str, log_root: Path):
    log_root = Path(log_root)

    # if caller passed the global root, resolve scenario dir automatically
    sc_dir = log_root / scenario_key.lower()
    if sc_dir.exists():
        log_root = sc_dir
    
    # extract scenario config
    sc = SCENARIOS[scenario_key]
    expected_sources = get_expected_sources_for_scenario(sc)

    # ---- multi-source run -----
    multi = run_pipeline_for_sources(expected_sources, log_root)

    # --- single-source runs ---
    single_rows = []
    single_outputs = {}
    for s in expected_sources:
        out = run_pipeline_for_sources([s], log_root)
        single_outputs[s] = out
        single_rows.append({
            "scenario": sc.id,
            "name": sc.name,
            "mode": "single",
            "source_set": s,
            **out["recon"],
            "failure_keys": sorted(list(out["failures"].keys())) if isinstance(out["failures"], dict) else [],
        })
    
    # best single-source by recon
    single_df = pd.DataFrame(single_rows)
    if len(single_df):
        best_single = single_df.sort_values(["reconstructability", "n_chain_steps", "n_tagged_steps"], ascending=False).head(1)
        best_single_row = best_single.iloc[0].to_dict()
    else:
        best_single_row = {}

    # multi row
    multi_row = {
        "scenario": sc.id,
        "name": sc.name,
        "mode": "multi",
        "source_set": "+".join(expected_sources),
        **multi["recon"],
        "failure_keys": sorted(list(multi["failures"].keys())) if isinstance(multi["failures"], dict) else [],
    }

    # --- Gap explanation helpers (for Q2/Q3) ---
    # Which steps appear in multi-source scenarios but are generally missing in single-source scenarios? (This demonstrates the complementarity of multi-source scenarios)
    multi_steps = set(multi["recon"]["observed_steps"])
    step_presence = {}
    for s, out in single_outputs.items():
        step_presence[s] = set(out["recon"]["observed_steps"])

    step_missing_in_single = {}
    for st in sorted(multi_steps):
        missing_sources = [s for s in expected_sources if st not in step_presence.get(s, set())]
        if missing_sources:
            step_missing_in_single[st] = missing_sources

    # Which failures are most frequent in single-source scenarios but mitigated by multi-source scenarios? (Reflecting mitigation)

    # Here, we simply use the difference of the key in failure_taxonomy; you can further expand the counts.
    def failure_keys(x): 
        return set(x.keys()) if isinstance(x, dict) else set()

    multi_fail = failure_keys(multi["failures"])
    single_fail_union = set()
    for s in expected_sources:
        single_fail_union |= failure_keys(single_outputs[s]["failures"])

    mitigated_failures = sorted(list(single_fail_union - multi_fail))

    summary = {
        "scenario": sc.id,
        "name": sc.name,
        "expected_sources": expected_sources,
        "multi_reconstructability": multi["recon"]["reconstructability"],
        "best_single_source": best_single_row.get("source_set"),
        "best_single_reconstructability": best_single_row.get("reconstructability", 0.0),
        "multi_minus_best_single": multi["recon"]["reconstructability"] - best_single_row.get("reconstructability", 0.0),
        "steps_missing_in_single": step_missing_in_single,   
        "failures_mitigated_by_multi": mitigated_failures, 

        # ---- CSA-2 placeholders (leave for later) ----
        "expected_payload_ttps": None,
        "expected_c2_ttps": None,
    }

    return {
        "multi": multi,
        "single": single_outputs,
        "single_df": single_df,
        "multi_row": multi_row,
        "summary": summary,
    }



In [5]:
def pretty_three_questions(summary: dict) -> str:
    sc = summary["scenario"]
    name = summary["name"]
    multi_r = summary["multi_reconstructability"]
    best_s = summary["best_single_source"]
    best_r = summary["best_single_reconstructability"]
    delta = summary["multi_minus_best_single"]

    # Q1: end-to-end reconstructability
    q1 = (
        f"[Q1] {sc} ({name}) — End-to-end reconstructability under multi-source telemetry: "
        f"reconstructability={multi_r:.2f}. "
        f"Best single-source baseline ({best_s})={best_r:.2f} "
        f"(multi − best_single = {delta:+.2f})."
    )

    # Q2: where/why single-source fails (proxy via steps missing in single-source runs)
    missing = summary["steps_missing_in_single"]
    top_missing = list(missing.items())[:8]  # keep it readable

    if top_missing:
        q2_lines = [
            f"  - step={st} is absent in single-source runs for: {srcs}"
            for st, srcs in top_missing
        ]
        q2 = (
            "[Q2] Single-source failure points — examples of steps observable in the multi-source run "
            "but missing under individual sources:\n"
            + "\n".join(q2_lines)
        )
    else:
        q2 = (
            "[Q2] Single-source failure points — no clear step-level omissions detected "
            "(this can happen when the chain is short, when sources have overlapping visibility, "
            "or when labeling evidence is sparse)."
        )

    # Q3: how multi-source mitigates failures (proxy via failure categories that disappear in multi-source)
    mitigated = summary["failures_mitigated_by_multi"]

    if mitigated:
        q3 = (
            "[Q3] Multi-source mitigation — failure categories observed in at least one single-source run "
            "but not present in the multi-source run:\n"
            + "\n".join([f"  - {x}" for x in mitigated[:10]])
        )
    else:
        q3 = (
            "[Q3] Multi-source mitigation — no clear difference in failure keys "
            "(consider refining this with step-level failure counts and/or explicit observability-gap signals)."
        )

    return q1 + "\n" + q2 + "\n" + q3



In [6]:
LOG_ROOT = Path.cwd().parent.joinpath("sanidata")

print("LOG_ROOT:", LOG_ROOT.resolve())
print("LOG_ROOT exists:", LOG_ROOT.exists())

all_files = [p for p in LOG_ROOT.rglob("*") if p.is_file()]
print("n_files under LOG_ROOT:", len(all_files))
print("sample:", [str(p.relative_to(LOG_ROOT)) for p in all_files[:50]])


LOG_ROOT: /Users/zhuoran/Projects/SSCMDataset/sanidata
LOG_ROOT exists: True
n_files under LOG_ROOT: 121
sample: ['.DS_Store', 'sc1/attack_navigator_cmds.json', 'sc1/attack_navigator_tasks.json', 'sc1/payload_ttps_2.json', 'sc1/payload_ttps_3.json', 'sc1/.DS_Store', 'sc1/payload_ttps_0.json', 'sc1/payload_ttps_1.json', 'sc1/windows/azure_port.csv', 'sc1/windows/azure_events.parquet', 'sc1/windows/azure_process.parquet', 'sc1/windows/azure_port.parquet', 'sc1/windows/azure_conn.parquet', 'sc1/windows/azure_process.csv', 'sc1/windows/azure_security.csv', 'sc1/windows/azure_conn.csv', 'sc1/windows/azure_perf.parquet', 'sc1/windows/azure_security.parquet', 'sc1/windows/azure_perf.csv', 'sc1/windows/azure_events.csv', 'sc6/attack_navigator_cmds.json', 'sc6/attack_navigator_tasks.json', 'sc6/payload_ttps_2.json', 'sc6/payload_ttps_3.json', 'sc6/.DS_Store', 'sc6/payload_ttps_4.json', 'sc6/payload_ttps_0.json', 'sc6/payload_ttps_1.json', 'sc6/victim/azure_events.parquet', 'sc6/victim/custom_au

In [7]:

sc1_dir = LOG_ROOT / "sc1"

print("files conn:", list(sc1_dir.rglob("azure_conn.csv")))
print("files conn windows:", list(sc1_dir.rglob("windows/azure_conn.csv")))

df = parse_source("azure_conn", sc1_dir)
print("parsed rows:", len(df))
print("cols:", df.columns.tolist()[:20] if len(df) else None)

files conn: [PosixPath('/Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/azure_conn.csv')]
files conn windows: [PosixPath('/Users/zhuoran/Projects/SSCMDataset/sanidata/sc1/windows/azure_conn.csv')]
parsed rows: 5746
cols: ['ts', 'host', 'source', 'event_type', 'action', 'subject', 'object', 'src', 'dst', 'raw', 'source_kind', 'source_file']


In [8]:
# ---- DEBUG: Azure/Syslog ingress for SC1 + SC2 ----

SC_DIRS = {
    "sc1": LOG_ROOT / "sc1",
    "sc2": LOG_ROOT / "sc2",
}

AZ_SYS_KEYS = ["syslog", "azure_conn", "azure_process", "azure_security", "azure_events", "azure_port"]

# filename patterns to brute-force inspect (independent from parse_source)
PATTERNS = {
    "syslog": ["azure_syslog.csv", "*syslog*.csv", "*syslog*.log"],
    "azure_conn": ["azure_conn.csv", "*azure*conn*.csv", "*azure_conn*.csv"],
    "azure_process": ["azure_process.csv", "*azure*process*.csv", "*azure_process*.csv"],
    "azure_security": ["azure_security.csv", "*azure*security*.csv", "*azure_security*.csv"],
    "azure_events": ["azure_events.csv", "*azure*events*.csv", "*azure_events*.csv"],
    "azure_port": ["azure_port.csv", "*azure*port*.csv", "*azure_port*.csv"],
}

rows = []

for sc, sc_dir in SC_DIRS.items():
    print("\n" + "=" * 90)
    print(f"[{sc}] SC_DIR = {sc_dir} | exists={sc_dir.exists()}")
    if not sc_dir.exists():
        rows.append({"scenario": sc, "check": "sc_dir_exists", "ok": False, "detail": str(sc_dir)})
        continue

    # 1) brute-force file presence check (rglob patterns)
    for k in AZ_SYS_KEYS:
        pats = PATTERNS.get(k, [])
        hits = []
        for pat in pats:
            hits.extend(list(sc_dir.rglob(pat)))
        hits = sorted({p for p in hits if p.is_file()})

        print(f"\n[{sc}] file-scan key={k}  hits={len(hits)}")
        for p in hits[:8]:
            print("  -", p.relative_to(sc_dir))
        rows.append({
            "scenario": sc,
            "check": "file_scan",
            "source_key": k,
            "ok": len(hits) > 0,
            "n_files": len(hits),
            "sample_files": "; ".join([str(p.relative_to(sc_dir)) for p in hits[:5]]),
        })

    # 2) parse_source check (the real ingress)
    for k in AZ_SYS_KEYS:
        try:
            df = parse_source(k, sc_dir)
            n = 0 if df is None else len(df)
            cols = [] if df is None else list(df.columns)
            has_ts = ("ts" in cols)
            ts_dtype = None
            ts_head = None
            if df is not None and n > 0 and has_ts:
                ts_dtype = str(df["ts"].dtype)
                ts_head = df["ts"].head(3).tolist()

            print(f"\n[{sc}] parse_source key={k}  n_events={n}  has_ts={has_ts}")
            if n > 0:
                print("  cols(head):", cols[:20])
                if has_ts:
                    print("  ts dtype:", ts_dtype, " ts head:", ts_head)
                # show a tiny sample row (to confirm schema)
                print("  sample row:", df.iloc[0].to_dict())

            rows.append({
                "scenario": sc,
                "check": "parse_source",
                "source_key": k,
                "ok": n > 0,
                "n_events": n,
                "has_ts": has_ts,
                "ts_dtype": ts_dtype,
                "cols": ", ".join(cols[:25]),
            })

        except Exception as e:
            print(f"\n[{sc}] parse_source key={k}  ERROR: {repr(e)}")
            rows.append({
                "scenario": sc,
                "check": "parse_source",
                "source_key": k,
                "ok": False,
                "error": repr(e),
            })

debug_df = pd.DataFrame(rows)

# summary view
print("\n" + "=" * 90)
print("SUMMARY (parse_source):")
display(
    debug_df[debug_df["check"] == "parse_source"]
    .sort_values(["scenario", "source_key"])
    .reset_index(drop=True)
)

print("\nSUMMARY (file_scan):")
display(
    debug_df[debug_df["check"] == "file_scan"]
    .sort_values(["scenario", "source_key"])
    .reset_index(drop=True)
)



[sc1] SC_DIR = /Users/zhuoran/Projects/SSCMDataset/sanidata/sc1 | exists=True

[sc1] file-scan key=syslog  hits=0

[sc1] file-scan key=azure_conn  hits=1
  - windows/azure_conn.csv

[sc1] file-scan key=azure_process  hits=1
  - windows/azure_process.csv

[sc1] file-scan key=azure_security  hits=1
  - windows/azure_security.csv

[sc1] file-scan key=azure_events  hits=1
  - windows/azure_events.csv

[sc1] file-scan key=azure_port  hits=1
  - windows/azure_port.csv

[sc1] parse_source key=syslog  n_events=0  has_ts=False

[sc1] parse_source key=azure_conn  n_events=5746  has_ts=True
  cols(head): ['ts', 'host', 'source', 'event_type', 'action', 'subject', 'object', 'src', 'dst', 'raw', 'source_kind', 'source_file']
  ts dtype: datetime64[ns]  ts head: [Timestamp('2025-04-22 13:03:00.020000'), Timestamp('2025-04-22 13:03:00.020000'), Timestamp('2025-04-22 13:03:00.020000')]
  sample row: {'ts': Timestamp('2025-04-22 13:03:00.020000'), 'host': 'HOST_050', 'source': 'azure_conn', 'event_typ

,scenario,check,source_key,ok,n_files,sample_files,n_events,has_ts,ts_dtype,cols
0,sc1,parse_source,azure_conn,True,NaN,NaN,5746.0,True,datetime64[ns],"ts, host, source, event_type, action, subject,..."
1,sc1,parse_source,azure_events,True,NaN,NaN,1000.0,True,datetime64[ns],"ts, host, source, event_type, action, subject,..."
2,sc1,parse_source,azure_port,True,NaN,NaN,1109.0,True,datetime64[ns],"ts, host, source, event_type, action, subject,..."
3,sc1,parse_source,azure_process,True,NaN,NaN,345.0,True,datetime64[ns],"ts, host, source, event_type, action, subject,..."
4,sc1,parse_source,azure_security,True,NaN,NaN,334.0,True,datetime64[ns],"ts, host, source, event_type, action, subject,..."
5,sc1,parse_source,syslog,False,NaN,NaN,0.0,False,None,
6,sc2,parse_source,azure_conn,False,NaN,NaN,0.0,False,None,
7,sc2,parse_source,azure_events,True,NaN,NaN,44246.0,True,datetime64[ns],"ts, host, source, event_type, action, subject,..."
8,sc2,parse_source,azure_port,False,NaN,NaN,0.0,False,None,
9,sc2,parse_source,azure_process,False,NaN,NaN,0.0,False,None,



SUMMARY (file_scan):


,scenario,check,source_key,ok,n_files,sample_files,n_events,has_ts,ts_dtype,cols
0,sc1,file_scan,azure_conn,True,1.0,windows/azure_conn.csv,NaN,NaN,NaN,NaN
1,sc1,file_scan,azure_events,True,1.0,windows/azure_events.csv,NaN,NaN,NaN,NaN
2,sc1,file_scan,azure_port,True,1.0,windows/azure_port.csv,NaN,NaN,NaN,NaN
3,sc1,file_scan,azure_process,True,1.0,windows/azure_process.csv,NaN,NaN,NaN,NaN
4,sc1,file_scan,azure_security,True,1.0,windows/azure_security.csv,NaN,NaN,NaN,NaN
5,sc1,file_scan,syslog,False,0.0,,NaN,NaN,NaN,NaN
6,sc2,file_scan,azure_conn,False,0.0,,NaN,NaN,NaN,NaN
7,sc2,file_scan,azure_events,True,1.0,azure_events.csv,NaN,NaN,NaN,NaN
8,sc2,file_scan,azure_port,False,0.0,,NaN,NaN,NaN,NaN
9,sc2,file_scan,azure_process,False,0.0,,NaN,NaN,NaN,NaN


In [10]:
sc1_dir = LOG_ROOT / "sc1"
print("SC1 dir:", sc1_dir, "exists:", sc1_dir.exists())

hits = sorted([p for p in sc1_dir.rglob("zeek_*.csv") if p.is_file()])
print("zeek_*.csv hits:", len(hits))
for p in hits[:20]:
    print(" -", p.relative_to(sc1_dir))

z = parse_source("zeek", sc1_dir)
print("\nparse_source('zeek') rows:", len(z))
if len(z) and "source_file" in z.columns:
    sf = z["source_file"].astype(str)
    print("unique source_file count:", sf.nunique())
    print("top source_files:")
    print(sf.value_counts().head(10))
    print("\nany source_file outside sc1_dir?",
          (~sf.str.startswith(str(sc1_dir.resolve()))).any())


SC1 dir: /Users/zhuoran/Projects/SSCMDataset/sanidata/sc1 exists: True
zeek_*.csv hits: 0

parse_source('zeek') rows: 0


In [9]:
# ---- run (scenario-local root) ----
scenario_keys = list(SCENARIOS.keys())
all_rows = []
all_summaries = {}

for k in scenario_keys:
    sc_dir = LOG_ROOT / k.lower()  
    res = analyze_scenario(k, sc_dir)  
    all_rows.append(res["multi_row"])
    if len(res["single_df"]):
        all_rows.extend(res["single_df"].to_dict("records"))
    all_summaries[k] = res["summary"]

result_df = pd.DataFrame(all_rows)
result_df.sort_values(["scenario", "mode", "reconstructability"], ascending=[True, True, False], inplace=True)
result_df


/Users/zhuoran/Projects/SSCMDataset/ana/labeling/tagger.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = rule_mask & blob.str.contains(rx, na=False)
/Users/zhuoran/Projects/SSCMDataset/ana/labeling/tagger.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = rule_mask & blob.str.contains(rx, na=False)
/Users/zhuoran/Projects/SSCMDataset/ana/labeling/tagger.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = rule_mask & blob.str.contains(rx, na=False)
/Users/zhuoran/Projects/SSCMDataset/ana/labeling/tagger.py:70: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  hit = rule_mask & blob.str.contains(rx, na=False)
/Users/z

,scenario,name,mode,source_set,n_events,n_tagged_steps,n_chain_steps,reconstructability,observed_steps,chain_steps,failure_keys
0,SC1,Stegano,multi,azure_conn+azure_process+azure_security+azure_...,8534,1,1,1.0,[OUTBOUND_CONN],[OUTBOUND_CONN],[]
26,SC1,Stegano,multi,auditd+auth+suricata+syslog+zeek,89910,2,2,1.0,"[AUTH, OUTBOUND_CONN]","[AUTH, OUTBOUND_CONN]",[]
1,SC1,Stegano,single,azure_conn,5746,1,1,1.0,[OUTBOUND_CONN],[OUTBOUND_CONN],[]
27,SC1,Stegano,single,auditd,14763,1,1,1.0,[AUTH],[AUTH],[]
29,SC1,Stegano,single,suricata,58880,1,1,1.0,[OUTBOUND_CONN],[OUTBOUND_CONN],[]
31,SC1,Stegano,single,zeek,6462,1,1,1.0,[OUTBOUND_CONN],[OUTBOUND_CONN],[]
2,SC1,Stegano,single,azure_process,345,0,0,0.0,[],[],[]
3,SC1,Stegano,single,azure_security,334,0,0,0.0,[],[],[]
4,SC1,Stegano,single,azure_events,1000,0,0,0.0,[],[],[]
5,SC1,Stegano,single,azure_port,1109,0,0,0.0,[],[],[]
